# 🦀 Robotic crab: train the walking policy on a GPU

1. **Runtime → Change runtime type → T4 GPU** (free tier is fine).
2. On your PC run `python training/make_colab_bundle.py` and upload `dist/crab_bundle.zip` when asked below.
3. Run the cells top to bottom. The full run takes roughly 1–2 h on a T4; keep the tab open.
4. The last cell downloads the trained policy. Unzip it into `training/checkpoints/` on your PC and run
   `python sim/walk.py --view --policy training/checkpoints/colab-full`.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
%%capture
# versions verified together; see requirements-train.txt
!pip install -q "jax[cuda12]==0.9.2" "jaxlib==0.9.2" "flax==0.12.6" "optax==0.2.9" "orbax-checkpoint==0.12.11" "brax==0.14.2" "mujoco==4.0.1" "mujoco-mjx==4.0.1" pillow matplotlib

In [ ]:
from google.colab import files
import zipfile, os
up = files.upload()                      # choose dist/crab_bundle.zip
zipfile.ZipFile(next(iter(up))).extractall('/content/crab')
%cd /content/crab
os.environ['MUJOCO_GL'] = 'egl'         # headless rendering for the GIF
!ls

### Smoke test (about 1–3 min, mostly JIT compilation)

In [ ]:
!python training/train.py --preset smoke --run colab-smoke

### Full training run

In [ ]:
!python training/train.py --preset full --run colab-full

### Learning curve

In [ ]:
import json, matplotlib.pyplot as plt
rows = [json.loads(l) for l in open('training/checkpoints/colab-full/progress.jsonl')]
plt.plot([r['step'] for r in rows], [r.get('eval/episode_reward') for r in rows], marker='o')
plt.xlabel('environment steps'); plt.ylabel('eval episode reward'); plt.grid(alpha=.3); plt.show()

### Watch it: same test and demo harness as the scripted gait

In [ ]:
!python sim/walk.py --policy training/checkpoints/colab-full
!python sim/walk.py --gif --policy training/checkpoints/colab-full
from IPython.display import Image
Image(open('docs/images/walk_policy.gif','rb').read())

### Download the trained policy

In [ ]:
!cd training/checkpoints && zip -qr /content/colab-full.zip colab-full
files.download('/content/colab-full.zip')